In [1]:
# %pip install light-curve
# %pip install nested-pandas --upgrade

In [2]:
import lsdb
import numpy as np
import matplotlib.pyplot as plt
from dask.distributed import Client
import light_curve as licu
import pyarrow as pa
from nested_pandas.utils import count_nested
from nested_pandas import read_parquet
from lsdb.core.search.region_search import MOCSearch

In [3]:
DDF = True

In [4]:
lsdb.__version__

In [5]:
import nested_pandas
nested_pandas.__version__

In [6]:
client = Client(n_workers=1, memory_limit="10 GiB", threads_per_worker=1)
display(client)
# client = Client(n_workers=10, memory_limit="10 GiB", threads_per_worker=1)
# display(client)

In [7]:
dia_object_path = '/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_6/dia_object_collection'
object_path = '/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_6/object_collection'

In [8]:
# DDF fields
# fields = {
#     "ECDFS": (53.13, -28.10),  # Extended Chandra Deep Field South
#     "EDFS": (59.10, -48.73),  # Euclid Deep Field South
#     "Rubin_SV_38_7": (37.86, 6.98),  # Low Ecliptic Latitude Field
#     "Rubin_SV_95_-25": (95.00, -25.00),  # Low Galactic Latitude Field
#     "47_Tuc": (6.02, -72.08),  # 47 Tuc Globular Cluster
#     "Fornax_dSph": (40.00, -34.45),  # Fornax Dwarf Spheroidal Galaxy
# }

# # Red for extragalactic fields, orange for dense fields
# field_styles = {
#     "ECDFS": ("red", "solid"),
#     "EDFS": ("red", "dashed"),
#     "Rubin_SV_38_7": ("red", "dotted"),
#     "Rubin_SV_95_-25": ("orange", "solid"),
#     "47_Tuc": ("orange", "dashed"),
#     "Fornax_dSph": ("orange", "dotted"),
# }

fields = {
    "ELAISS1": (9.45, -44.02),    # ELAIS-S1
    "XMM_LSS": (35.57, -4.82),    # XMM Large Scale Structure
    "ECDFS":   (52.98, -28.12),   # Extended Chandra Deep Field South
    "COSMOS":  (150.11, 2.23),    # COSMOS
    "EDFS_a":  (58.9, -49.32),    # Euclid Deep Field South a
    "EDFS_b":  (63.6, -47.60),    # Euclid Deep Field South b
}

field_styles = {
    "ELAISS1": ("red", "solid"),    # ELAIS-S1
    "XMM_LSS": ("red", "solid"),    # XMM Large Scale Structure
    "ECDFS":   ("red", "solid"),    # Extended Chandra Deep Field South
    "COSMOS":  ("red", "solid"),    # COSMOS
    "EDFS_a":  ("red", "solid"),    # Euclid Deep Field South a
    "EDFS_b":  ("red", "solid"),    # Euclid Deep Field South b
}

# Define a 4-degree search radius
radius_arcsec = 4 * 3600

# Create six cone searches
cones = {name: lsdb.ConeSearch(ra=ra, dec=dec, radius_arcsec=radius_arcsec) for name, (ra, dec) in fields.items()}

In [9]:
dia_object = lsdb.open_catalog(dia_object_path,
                               columns=["diaObjectId"])
fig, ax = dia_object.plot_pixels()
for name, cone in cones.items():
    color, linestyle = field_styles[name]
    cone.plot(ax=ax, ec=f"tab:{color}", linestyle=linestyle, label=name)
    ax.legend()

In [10]:
from hats.pixel_math import region_to_moc
from hats.inspection.visualize_catalog import plot_moc

max_depth = dia_object.hc_structure.get_max_coverage_order()

cone_moc_union = region_to_moc.cone_to_moc(ra=next(iter(fields.values()))[0], dec=next(iter(fields.values()))[1], radius_arcsec=radius_arcsec, max_depth=max_depth)
for key, value in list(fields.items())[1:]:
    cone_moc = region_to_moc.cone_to_moc(ra=value[0], dec=value[1], radius_arcsec=radius_arcsec, max_depth=max_depth)
    cone_moc_union = cone_moc_union.union(cone_moc)
plot_moc(cone_moc_union)

In [11]:
# select smaller area
if DDF:
    box = MOCSearch(cone_moc_union)
else:
    box = lsdb.BoxSearch(ra=[335,345], dec=[-20,-10])

In [12]:
dia_object = lsdb.open_catalog(dia_object_path,
                               columns=["diaObjectId",
                                        "diaObjectForcedSource.midpointMjdTai","diaObjectForcedSource.band",
                                        "diaObjectForcedSource.psfDiffFlux","diaObjectForcedSource.psfDiffFluxErr",
                                        "diaObjectForcedSource.psfDiffFlux_flag","diaObjectForcedSource.invalidPsfFlag",
                                        "diaSource.midpointMjdTai","diaSource.band",
                                        "diaSource.psfFlux","diaSource.psfFluxErr",
                                        "diaSource.templateFlux","diaSource.templateFluxErr",
                                        "diaSource.reliability","diaSource.psfFlux_flag"], 
                               search_filter=box)

In [13]:
objects = lsdb.open_catalog(object_path,columns=["refExtendedness","refSizeExtendedness","objectId"], search_filter=box)

In [14]:
galaxies = objects.query("refExtendedness > 0.7 & refSizeExtendedness > 0.7")

In [15]:
objects_around_gal = dia_object.crossmatch(galaxies, radius_arcsec=5)

In [16]:
objects_around_gal.plot_pixels()

In [17]:
objects_around_gal = objects_around_gal.query("_dist_arcsec > 0.5 and _dist_arcsec < 5")

In [18]:
objects_around_gal

In [19]:
def compute_nbands(band):
    nbands = len(np.unique(band))
    n_g = np.sum(band == "g")
    n_r = np.sum(band == "r")
    n_i = np.sum(band == "i")
    n_z = np.sum(band == "z")
    return nbands,n_g,n_r,n_i,n_z

def compute_all_columns(df):    
    cols = ["median_reliability","max_snr","ndet","nbands","n_g","n_r","n_i","n_z","dt",
            "diaSource_dia_object_lc.snr","diaSource_dia_object_lc.flux_diff_ratio","diaSource_dia_object_lc.flux_diff_ratio_abs"]
    if len(df) == 0:
        df = df.assign(**{name: np.array([], dtype=np.float32) for name in cols})

    else:
        # calculate median reliability
        df = df.map_rows(np.median, columns=["diaSource_dia_object_lc.reliability"], row_container="args", output_names=["median_reliability"], append_columns=True)
    
        # caclulate snr
        df["diaSource_dia_object_lc.snr"] = df["diaSource_dia_object_lc.psfFlux"]/df["diaSource_dia_object_lc.psfFluxErr"]
        
        # calculate max snr
        df = df.map_rows(np.max, columns=["diaSource_dia_object_lc.snr"], row_container="args", output_names=["max_snr"], append_columns=True)
        
        # calculate psfflux difference between template and difference image
        df["diaSource_dia_object_lc.flux_diff_ratio"] = (df["diaSource_dia_object_lc.psfFlux"] - df["diaSource_dia_object_lc.templateFlux"]) / df["diaSource_dia_object_lc.templateFlux"]
        df = df.map_rows(np.absolute, columns=["diaSource_dia_object_lc.flux_diff_ratio"], row_container="args", output_names=["diaSource_dia_object_lc.flux_diff_ratio_abs"], append_columns=True)
        
        # calculate number of detections
        df["ndet"] = df["diaSource_dia_object_lc"].len()
        
        # calculate number of bands
        df = df.map_rows(compute_nbands, columns=["diaSource_dia_object_lc.band"], row_container="args",
                         output_names = ["nbands","n_g","n_r","n_i","n_z"],append_columns=True)    
        
        # calculate detection range
        df = df.map_rows(np.ptp, columns=["diaSource_dia_object_lc.midpointMjdTai"], row_container="args", 
                         output_names = ["dt"], append_columns=True)
        # # calculate n per band
        # df = count_nested(df, "diaSource_dia_object_lc", by="band", join=True)
    return df

def filtering(df):
    # # filter reliability
    # df = df.query("median_reliability > 0.5")
            
    # # filter template and diff difference
    # df = df.query("diaSource_dia_object_lc.flux_diff_ratio_abs > 0.2")
    
    df = df.query(
                  # filter n detection
                  "ndet > 15 and "   
                  # filter maxsnr
                  "max_snr > 10 and "  
                  # filter dt
                  "(dt > 20 and dt < 100) and "
                  # filter nbands
                  "nbands >= 3 and " 
                  # filter n per band
                  "((n_r > 2 and n_i > 2) or "
                  "(n_g > 2 and n_r > 2) or "
                  "(n_i > 2 and n_z > 2))"
                 )
    return df

In [20]:
N_JOBS_PER_WORKER = -1
extractor = licu.Extractor(licu.BazinFit(algorithm='nuts-ceres',bands=["r","i"]), licu.ReducedChi2(bands=["r","i"]))
def fitbazin(df):
    df = df.query("diaObjectForcedSource_dia_object_lc.psfDiffFlux_flag == False and "
                  "diaObjectForcedSource_dia_object_lc.invalidPsfFlag == False")
    df_r = df.query("diaObjectForcedSource_dia_object_lc.band == 'r' or diaObjectForcedSource_dia_object_lc.band == 'i'")
    df_r["diaObjectForcedSource_dia_object_lc.t"] = (df_r["diaObjectForcedSource_dia_object_lc.midpointMjdTai"] - 60000).astype(np.float32)
    if len(df_r) == 0:
        df = df.assign(**{name: np.array([], dtype=np.float32) for name in extractor.names})
    else:
        features_r = extractor.many(pa.array(df_r["diaObjectForcedSource_dia_object_lc"]), 
                                    arrow_fields={"t": "t", "m": "psfDiffFlux", "sigma": "psfDiffFluxErr","band":"band"},
                                    n_jobs=N_JOBS_PER_WORKER, fill_value=np.nan)  # default -1 which uses all the CPU cores
        df = df.assign(**(dict(zip(extractor.names, features_r.T))))
    df = df.query("bazin_fit_reduced_chi2_r>0.1 and bazin_fit_reduced_chi2_r < 10.0")
    return df

In [21]:
objects_around_gal = objects_around_gal.map_partitions(compute_all_columns)

In [22]:
# objects_around_gal.head()

In [23]:
filtered = objects_around_gal.map_partitions(filtering)

In [24]:
# head = filtered.head()
# head

In [25]:
# filtered = lsdb.open_catalog("sncandid")
bazin = filtered.map_partitions(fitbazin)

In [ ]:
bazin.write_catalog("sncandid_w_bazin", overwrite=True)

In [ ]:
# def plot_lc(lc):
#     for f in lc["band"].unique():
#         lc_f = lc.loc[lc["band"]==f]
#         plt.errorbar(lc_f["midpointMjdTai"],lc_f["psfDiffFlux"],yerr=lc_f["psfDiffFlux"],fmt='o',label=f)
#         plt.legend()
#         plt.ylim((-5000,None))

In [ ]:
# for i in np.random.choice(len(head),size=5):
#     plt.subplot(1,1,1)
#     lc = head.iloc[i]["diaObjectForcedSource_dia_object_lc"]
#     lc = lc[["midpointMjdTai","band","psfDiffFlux","psfDiffFluxErr"]]
#     plot_lc(lc)
#     plt.show()

In [ ]:
# filtered.write_catalog("sncandid", overwrite=True)